# STREEVA — Area Risk Scoring Engine
## Full Pipeline Walkthrough
> **Purpose**: This notebook documents the complete methodology of the Area Risk Scoring Engine for inclusion in the STREEVA capstone project report. It covers data ingestion, spatial processing, feature engineering, scoring formula derivation, and qualitative validation.

---

### Table of Contents
1. [Why This Approach? — Data Availability in India](#section1)
2. [Dataset Exploration — NCRB Crime Data](#section2)
3. [H3 Hexagonal Grid System](#section3)
4. [Feature Engineering](#section4)
5. [Feature Visualization](#section5)
6. [Scoring Formula](#section6)
7. [Risk Heatmap](#section7)
8. [Time-of-Day Sensitivity Analysis](#section8)
9. [Example API Queries](#section9)
10. [Validation & Conclusions](#section10)

In [ ]:
# ── Setup: install missing packages if running standalone ─────────────────────
import sys, os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')
print(f'Python: {sys.version}')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import json
import math
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import h3
import folium
from shapely.geometry import Polygon
import geopandas as gpd

# Project modules
from app.config import get_scoring_weights, get_city_config
from app.core.h3_utils import latlng_to_h3, h3_to_polygon, fill_bbox_with_h3, haversine_distance_m
from app.core.time_multiplier import get_time_multiplier, get_all_multipliers
from app.scoring.weighted_scorer import WeightedRiskScorer
from app.scoring.base_scorer import FeatureSet

print('All imports successful.')

<a id='section1'></a>
## 1. Why This Approach? — Data Availability in India

### The Core Problem

In countries like the United States, police departments publish open datasets with per-incident crime data, including:
- Latitude / Longitude of the incident
- Crime type
- Date and time

This enables point-level geospatial crime modeling directly.

**In India, this data does not exist at the open-data level.** The NCRB (National Crime Records Bureau) under the Ministry of Home Affairs publishes annual crime statistics, but only at the **district or city level** — not per incident, not per neighborhood, not geocoded.

### What NCRB Provides

The dataset we have (`ncrb_ipc_crimes_2014.json`, sourced from data.gov.in) contains:
- **Unit**: District (e.g., 'Chennai', 'Coimbatore City')
- **Granularity**: Annual aggregate count per crime category
- **Crime categories**: 87 columns (Murder, Rape, Kidnapping, Theft, etc.)

It does **not** contain: coordinates of incidents, sub-district zones, police station beats, or time-of-day breakdowns.

### Our Two-Layer Solution

```
Layer 1 — Macro Baseline (NCRB)
  ↓ How dangerous is Chennai district relative to Tamil Nadu?
  ↓ Applied uniformly as a city-wide offset

Layer 2 — Hyperlocal Proxy Risk (OSM + Places + WorldPop)
  ↓ How does THIS street block compare to others in Chennai?
  ↓ Primary signal — most engineering effort
```

This is the **correct and honest approach** for the Indian data context.

<a id='section2'></a>
## 2. Dataset Exploration — NCRB Crime Data

In [ ]:
# Load the raw NCRB dataset
with open('../data/raw/ncrb_ipc_crimes_2014.json', encoding='utf-8') as f:
    raw = json.load(f)

fields = {item['id']: item['label'] for item in raw['fields']}
data = raw['data']

# Build a DataFrame
field_labels = [item['label'] for item in raw['fields']]
df = pd.DataFrame(data, columns=field_labels)

# Convert numeric columns
num_cols = field_labels[3:]  # Skip State, District, Year
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

print(f'Total rows: {len(df)}')
print(f'Columns: {len(df.columns)}')
print(f'States/UTs: {df["States/UTs"].nunique()}')
print(f'Districts: {df["District"].nunique()}')
print(f'Years: {df["Year"].unique()}')
df.head(3)

In [ ]:
# ── Tamil Nadu deep dive ──────────────────────────────────────────────────────
tn = df[(df['States/UTs'] == 'Tamil Nadu') & (~df['District'].isin(['Total','Other Units','Cyber Cell']))].copy()

# Violent crimes subset (matches ingest_ncrb.py selection)
violent_cols = [
    'Murder', 'Attempt to commit Murder', 'Rape', 'Attempt to commit Rape',
    'Kidnapping & Abduction_Total', 'Dacoity', 'Robbery', 'Grievous Hurt', 'Hurt',
    'Acid attack', 'Assault on Women with intent to outrage her Modesty',
    'Sexual Harassment', 'Dowry Deaths', 'Extortion', 'HumanTrafficking'
]
# Keep only columns that exist in our dataset
violent_cols = [c for c in violent_cols if c in df.columns]

tn['violent_crimes'] = tn[violent_cols].sum(axis=1)
tn_sorted = tn[['District','violent_crimes','Total Cognizable IPC crimes']].sort_values('violent_crimes', ascending=False)

print('Tamil Nadu Districts — Violent vs Total IPC Crimes (2014):')
print(tn_sorted.to_string(index=False))

In [ ]:
# ── Visualization: TN Districts violent crime ranking ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Chart 1: Violent crimes by district
plot_data = tn_sorted.head(15)
colors = ['#e74c3c' if d == 'Chennai' else '#3498db' for d in plot_data['District']]
axes[0].barh(plot_data['District'], plot_data['violent_crimes'], color=colors)
axes[0].set_xlabel('Violent Crime Count (2014)', fontsize=12)
axes[0].set_title('Tamil Nadu Districts — Violent Crimes\n(Red = Chennai)', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Chart 2: Proportion of violent vs other IPC
chennai = tn[tn['District'] == 'Chennai'].iloc[0]
violent = chennai['violent_crimes']
rash_driving = chennai.get('Incidence of Rash Driving', 0)
other_ipc = chennai['Total Cognizable IPC crimes'] - violent - rash_driving

wedge_labels = [f'Violent Crimes\n({int(violent):,})', 
                f'Rash Driving\n({int(rash_driving):,})',
                f'Other IPC\n({int(other_ipc):,})']
wedge_sizes = [violent, rash_driving, other_ipc]
wedge_colors = ['#e74c3c', '#f39c12', '#95a5a6']
axes[1].pie(wedge_sizes, labels=wedge_labels, colors=wedge_colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title(f'Chennai IPC Crime Composition (2014)\nTotal: {int(chennai["Total Cognizable IPC crimes"]):,}',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/tn_crime_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey insight: Rash Driving accounts for ~50% of Chennai total IPC.')
print('We use violent crimes only (murder, rape, assault etc.) as the baseline — not total IPC.')

In [ ]:
# ── Run NCRB ingestion (if not already done) ──────────────────────────────────
import subprocess
import os

lookup_path = '../data/processed/ncrb_district_lookup.json'
if not os.path.exists(lookup_path):
    print('Running NCRB ingestion pipeline...')
    result = subprocess.run(['python', '../scripts/ingest_ncrb.py'], 
                           capture_output=True, text=True, cwd=project_root)
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
else:
    print(f'NCRB lookup already exists: {lookup_path}')

# Load and display
with open(lookup_path) as f:
    ncrb_lookup = json.load(f)

chennai_data = ncrb_lookup.get('Chennai', {})
print(f'\nChennai crime baseline data:')
for k, v in chennai_data.items():
    if k != 'normalization':
        print(f'  {k}: {v}')

<a id='section3'></a>
## 3. H3 Hexagonal Grid System

### Why Hexagons?

We use **Uber's H3** hexagonal grid system instead of a naive lat/lng grid.

| Property | Square Grid | H3 Hexagonal Grid |
|----------|-------------|-------------------|
| Distance to all neighbors | Non-uniform (diagonal > orthogonal) | **Equal** (all 6 neighbors equidistant) |
| Area distortion at latitude | Yes | **No** (equal-area at same resolution) |
| Multi-resolution hierarchy | Complex | **Native** (parent/child cells) |
| Cache key | (lat_bin, lng_bin) tuple | **Single 64-bit index string** |

### Resolution 9 — Why?

| Resolution | Edge Length | Area | Typical Size |
|-----------|-------------|------|-------------|
| 7 | ~1,406 m | ~5.2 km² | Neighborhood |
| 8 | ~531 m | ~0.74 km² | Several blocks |
| **9** | **~201 m** | **~0.11 km²** | **~1 city block** |
| 10 | ~76 m | ~0.01 km² | Half a block |

Resolution 9 gives us **city-block granularity** in Chennai — small enough to distinguish between a main road and a nearby service lane, large enough to have meaningful OSM and Places data in each cell.

In [ ]:
# ── H3 grid visualization over Chennai ───────────────────────────────────────
from app.core.h3_utils import fill_bbox_with_h3, h3_to_polygon, h3_edge_length_m

city_config = get_city_config()
bbox = city_config.bbox

# Show H3 resolution properties
print('H3 Resolution Comparison:')
print(f'  Resolution 7: edge={h3_edge_length_m(7):.0f}m (neighborhood scale)')
print(f'  Resolution 8: edge={h3_edge_length_m(8):.0f}m (multi-block scale)')
print(f'  Resolution 9: edge={h3_edge_length_m(9):.0f}m ← CHOSEN (city block scale)')
print(f'  Resolution 10: edge={h3_edge_length_m(10):.0f}m (sub-block scale)')

# Fill a SMALL area with H3 cells for visualization
viz_bbox = {'south': 13.04, 'west': 80.24, 'north': 13.08, 'east': 80.29}
cells_res9 = fill_bbox_with_h3(**viz_bbox, resolution=9)
cells_res8 = fill_bbox_with_h3(**viz_bbox, resolution=8)

print(f'\nSample area ({viz_bbox["south"]}-{viz_bbox["north"]}N, {viz_bbox["west"]}-{viz_bbox["east"]}E):')
print(f'  Resolution 8 cells: {len(cells_res8)}')
print(f'  Resolution 9 cells: {len(cells_res9)}')

In [ ]:
# ── Interactive Folium map: H3 grid over Chennai sample area ──────────────────
m = folium.Map(
    location=[13.06, 80.265],
    zoom_start=14,
    tiles='CartoDB positron'
)

# Draw resolution 9 hexagons
for cell in list(cells_res9)[:200]:  # Limit for display
    poly = h3_to_polygon(cell)
    coords = [[lat, lng] for lng, lat in poly.exterior.coords]
    folium.Polygon(
        locations=coords,
        color='#3498db',
        weight=1,
        fill=True,
        fill_color='#3498db',
        fill_opacity=0.1,
        tooltip=f'H3 Cell: {cell}'
    ).add_to(m)

# Add key landmarks
landmarks = [
    {'name': 'Marina Beach', 'lat': 13.0500, 'lng': 80.2824},
    {'name': 'T. Nagar', 'lat': 13.0418, 'lng': 80.2341},
]
for lm in landmarks:
    folium.Marker([lm['lat'], lm['lng']], popup=lm['name'],
                  icon=folium.Icon(color='red', icon='info-sign')).add_to(m)

m.save('../data/processed/h3_grid_map.html')
print('Map saved: data/processed/h3_grid_map.html')
print('Open this file in a browser to see the interactive H3 grid.')
m

<a id='section4'></a>
## 4. Feature Engineering

For each H3 cell we compute 6 feature scores (all normalized to 0–100, where higher = riskier).

### Feature 1: Crime Baseline (NCRB)
- **Definition**: District-level violent crime count, normalized across Tamil Nadu districts  
- **Formula**: `(violent_crimes_district - min_TN) / (max_TN - min_TN) × 100`
- **Units**: Dimensionless score 0–100
- **Limitation**: Same value for every H3 cell in Chennai — it is a macro-level prior, not hyperlocal

### Feature 2: Isolation Score (OSM)
- **Definition**: How road-isolated a location is — higher = more isolated = riskier
- **Formula**: `0.50 × road_type_score + 0.30 × (1 - road_density_normalized) + 0.20 × (1 - junction_density_normalized)`
- **Road type scores**: motorway=5, primary=15, secondary=25, residential=50, service=70, track=85
- **Source**: OpenStreetMap via OSMnx

### Feature 3: Commercial Density (Google Places)
- **Definition**: Count of OPERATIONAL businesses within 500m (inverted)
- **Formula**: `100 - clamp(count / 20 × 100, 0, 100)`
- **Theoretical basis**: Jane Jacobs (1961) — "eyes on the street" reduce crime
- **Field mask**: Only `businessStatus, types, location` — no reviews/ratings

### Feature 4 & 5: Police / Hospital Distance (Google Places)
- **Definition**: Sigmoid of distance to nearest facility
- **Formula**: `100 / (1 + exp(-k × (distance_m - midpoint)))`
  - Police: midpoint=2500m, k=0.001
  - Hospital: midpoint=3000m, k=0.001

### Feature 6: Population Density (WorldPop)
- **Definition**: Pixel-level population count (inverted — dense = safer)
- **Formula**: `(1 - (pop - min) / (max - min)) × 100`
- **Source**: WorldPop 2020, 100m resolution GeoTIFF

In [ ]:
# ── Feature normalization curves ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Police/Hospital distance sigmoid
distances = np.linspace(0, 6000, 300)
police_scores = 100 / (1 + np.exp(-0.001 * (distances - 2500)))
hospital_scores = 100 / (1 + np.exp(-0.001 * (distances - 3000)))

axes[0].plot(distances, police_scores, color='#e74c3c', linewidth=2.5, label='Police (mid=2500m)')
axes[0].plot(distances, hospital_scores, color='#3498db', linewidth=2.5, label='Hospital (mid=3000m)')
axes[0].axvline(x=2500, color='#e74c3c', linestyle='--', alpha=0.4)
axes[0].axvline(x=3000, color='#3498db', linestyle='--', alpha=0.4)
axes[0].axhline(y=50, color='gray', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Distance to Facility (m)', fontsize=11)
axes[0].set_ylabel('Risk Score (0–100)', fontsize=11)
axes[0].set_title('Emergency Distance → Risk Score\n(Sigmoid Normalization)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 100)

# 2. Commercial density (inverted linear)
counts = np.arange(0, 25)
density_scores = np.maximum(0, 100 - counts / 20 * 100)

axes[1].bar(counts, density_scores, color=['#27ae60' if s < 50 else '#e74c3c' for s in density_scores], alpha=0.8)
axes[1].set_xlabel('Active Commercial POIs within 500m', fontsize=11)
axes[1].set_ylabel('Risk Score (0–100)', fontsize=11)
axes[1].set_title('Commercial Density → Risk Score\n(Inverted Linear — more shops = safer)', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(y=50, color='gray', linestyle=':', alpha=0.5)

# 3. Road type risk scores
road_types = ['motorway', 'primary', 'secondary', 'tertiary', 'residential', 'service', 'track']
weights_cfg = get_scoring_weights()
road_scores_cfg = weights_cfg.normalization['road_type_scores']
road_scores_vals = [road_scores_cfg[rt] for rt in road_types]
road_colors = ['#27ae60' if s < 40 else '#f39c12' if s < 65 else '#e74c3c' for s in road_scores_vals]

axes[2].barh(road_types, road_scores_vals, color=road_colors, alpha=0.9)
axes[2].set_xlabel('Road Type Risk Score', fontsize=11)
axes[2].set_title('OSM Highway Tag → Isolation Risk\n(Higher = More Isolated = Riskier)', fontsize=12, fontweight='bold')
axes[2].set_xlim(0, 100)
axes[2].invert_yaxis()
for i, (rt, val) in enumerate(zip(road_types, road_scores_vals)):
    axes[2].text(val + 1, i, str(val), va='center', fontsize=10)
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/feature_normalization_curves.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='section5'></a>
## 5. Feature Visualization — Demo Locations

In [ ]:
# ── Demo locations feature table (using simulated features for visualization) ─
# NOTE: Real features require live API calls. For the report, we show
# expected approximate feature values based on known location characteristics.

demo_data = {
    'Location': [
        'T. Nagar', 'Marina Beach', 'OMR IT Corridor',
        'Tambaram Market', 'VIT Chennai', 'Vandalur (Isolated)'
    ],
    'Road Type': ['primary', 'primary', 'primary', 'secondary', 'tertiary', 'service'],
    'Isolation Score': [18, 15, 22, 30, 45, 72],
    'Commercial Density': [5, 30, 25, 25, 55, 85],
    'Police Distance': [20, 25, 35, 30, 55, 65],
    'Hospital Distance': [25, 30, 40, 35, 50, 60],
    'Pop Density Score': [15, 20, 30, 25, 40, 70],
    'Crime Baseline': [54, 54, 54, 54, 54, 54],  # Same for all Chennai
}

df_demo = pd.DataFrame(demo_data)

# Compute raw risk score (day, no multiplier)
w = get_scoring_weights().weights
df_demo['Raw Score (Day)'] = (
    w['crime_baseline'] * df_demo['Crime Baseline'] +
    w['isolation'] * df_demo['Isolation Score'] +
    w['commercial_density'] * df_demo['Commercial Density'] +
    w['police_distance'] * df_demo['Police Distance'] +
    w['hospital_distance'] * df_demo['Hospital Distance'] +
    w['population_density'] * df_demo['Pop Density Score']
).round(1)

df_demo['Night Score (×1.4)'] = (df_demo['Raw Score (Day)'] * 1.4).clip(0, 100).round(1)

weights_cfg = get_scoring_weights()
df_demo['Classification (Day)'] = df_demo['Raw Score (Day)'].apply(weights_cfg.classify)

display_cols = ['Location', 'Isolation Score', 'Commercial Density', 'Police Distance',
                'Raw Score (Day)', 'Night Score (×1.4)', 'Classification (Day)']
print('Expected feature values and scores for demo locations:')
print('(Based on known location characteristics — actual values depend on live API data)')
print()
print(df_demo[display_cols].to_string(index=False))

In [ ]:
# ── Feature heatmap for demo locations ───────────────────────────────────────
feature_cols = ['Isolation Score', 'Commercial Density', 'Police Distance',
                'Hospital Distance', 'Pop Density Score', 'Crime Baseline']

heat_data = df_demo.set_index('Location')[feature_cols]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heat_data.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=100)

ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels([c.replace(' ', '\n') for c in feature_cols], fontsize=10)
ax.set_yticks(range(len(heat_data)))
ax.set_yticklabels(heat_data.index, fontsize=11)

for i in range(len(heat_data)):
    for j in range(len(feature_cols)):
        val = heat_data.values[i, j]
        text_color = 'white' if val > 65 or val < 30 else 'black'
        ax.text(j, i, str(int(val)), ha='center', va='center',
                fontsize=11, fontweight='bold', color=text_color)

plt.colorbar(im, ax=ax, label='Risk Score (0=Safe, 100=Risky)')
ax.set_title('Feature Risk Scores by Location\n(Green = Lower Risk, Red = Higher Risk)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../data/processed/feature_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='section6'></a>
## 6. Scoring Formula

### Formula Derivation

$$\text{RawScore} = \sum_{i} w_i \cdot f_i$$

$$\text{FinalScore} = \text{clamp}(\text{RawScore} \times \text{TimeMultiplier},\ 0,\ 100)$$

Where:

| Feature $f_i$ | Weight $w_i$ | Rationale |
|--------------|-------------|----------|
| Crime Baseline | 0.20 | Historical district prior; limited because it's city-wide |
| Isolation Score | **0.25** | Strongest single predictor — isolated locations lack natural surveillance |
| Commercial Density | 0.15 | Jane Jacobs "eyes on the street" — active businesses deter crime |
| Police Distance | 0.20 | Deterrence and response time |
| Hospital Distance | 0.10 | Affects outcomes, not probability — secondary factor |
| Population Density | 0.10 | Informal social control; correlated with commercial density |

All weights are in `configs/scoring_weights.yaml` — **no code change needed to tune them**.

In [ ]:
# ── Weight visualization ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

weights_cfg = get_scoring_weights()
weights = weights_cfg.weights

feature_names = {
    'crime_baseline': 'Crime Baseline\n(NCRB)',
    'isolation': 'Isolation Score\n(OSM Roads)',
    'commercial_density': 'Commercial Density\n(Google Places)',
    'police_distance': 'Police Distance\n(Google Places)',
    'hospital_distance': 'Hospital Distance\n(Google Places)',
    'population_density': 'Population Density\n(WorldPop)',
}

labels = [feature_names[k] for k in weights]
values = list(weights.values())
colors_pie = ['#e74c3c', '#e67e22', '#27ae60', '#3498db', '#9b59b6', '#1abc9c']

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    values, labels=labels, colors=colors_pie, autopct='%1.0f%%',
    startangle=90, pctdistance=0.75, textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0].set_title('Risk Score Feature Weights', fontsize=13, fontweight='bold')

# Bar chart of expected score contributions for isolated location at night
isolated_features = [50, 80, 85, 70, 65, 75]  # High-risk scenario
contributions = [w * f for w, f in zip(values, isolated_features)]
raw_score = sum(contributions)
final_score = min(raw_score * 1.40, 100)

short_labels = ['Crime\nBaseline', 'Isolation', 'Commercial\nDensity',
                'Police\nDistance', 'Hospital\nDistance', 'Population\nDensity']
bars = axes[1].bar(short_labels, contributions, color=colors_pie, alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_ylabel('Risk Contribution (pts)', fontsize=11)
axes[1].set_title(f'Example: Isolated Location at Night\nRaw Score: {raw_score:.1f} → Final (×1.40): {final_score:.1f}',
                  fontsize=12, fontweight='bold')
for bar, val in zip(bars, contributions):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, max(contributions) * 1.3)

plt.tight_layout()
plt.savefig('../data/processed/scoring_formula.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nWeights sum: {sum(values):.2f} (must be 1.0)')
print(f'Example isolated location score: {final_score:.1f}/100 → {weights_cfg.classify(final_score)}')

<a id='section7'></a>
## 7. Risk Heatmap

In [ ]:
# ── Simulated risk heatmap over Chennai ───────────────────────────────────────
# NOTE: A live heatmap requires API calls for each cell (takes ~10-60 mins for
# hundreds of cells). Here we demonstrate the visualization method with
# simulated data based on known Chennai geography.

# For your report, run the actual API and replace this with live scores.

import random
random.seed(42)

# Sample a smaller set of cells for visualization
sample_bbox = {'south': 12.97, 'west': 80.14, 'north': 13.12, 'east': 80.31}
sample_cells = list(fill_bbox_with_h3(**sample_bbox, resolution=9))
print(f'Sample cells for heatmap: {len(sample_cells)}')

# Assign simulated scores based on approximate Chennai geography
# (commercial areas = lower risk, periphery = higher risk)
cell_scores = []
for cell in sample_cells:
    lat, lng = h3.cell_to_latlng(cell)
    
    # Approximate: T.Nagar / central Chennai → safer
    dist_center = haversine_distance_m(lat, lng, 13.05, 80.24)
    base_score = min(25 + dist_center / 80, 80)
    
    # Add some variance
    noise = random.gauss(0, 8)
    score = max(5, min(95, base_score + noise))
    cell_scores.append((cell, lat, lng, score))

df_cells = pd.DataFrame(cell_scores, columns=['h3_cell', 'lat', 'lng', 'risk_score'])
print(f'Score range: {df_cells["risk_score"].min():.1f} – {df_cells["risk_score"].max():.1f}')
print(f'Mean score: {df_cells["risk_score"].mean():.1f}')

In [ ]:
# ── Folium heatmap ────────────────────────────────────────────────────────────
def score_to_color(score):
    """Map 0–100 score to a hex color: green → yellow → red."""
    if score <= 25: return '#27ae60'    # Green: Low
    elif score <= 50: return '#f39c12'  # Yellow: Medium
    elif score <= 75: return '#e67e22'  # Orange: High
    else: return '#e74c3c'              # Red: Critical

risk_map = folium.Map(location=[13.06, 80.24], zoom_start=13, tiles='CartoDB positron')

for _, row in df_cells.iterrows():
    poly = h3_to_polygon(row['h3_cell'])
    coords = [[lat, lng] for lng, lat in poly.exterior.coords]
    color = score_to_color(row['risk_score'])
    weights_cfg2 = get_scoring_weights()
    cls = weights_cfg2.classify(row['risk_score'])
    folium.Polygon(
        locations=coords,
        color=color,
        weight=0.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        tooltip=f"Score: {row['risk_score']:.1f} ({cls})"
    ).add_to(risk_map)

# Legend
legend_html = '''
<div style="position:fixed;bottom:30px;right:30px;background:white;padding:12px;border-radius:8px;box-shadow:2px 2px 6px rgba(0,0,0,0.3);font-size:13px;">
<b>Risk Level</b><br>
<span style="color:#27ae60">■</span> Low (0–25)<br>
<span style="color:#f39c12">■</span> Medium (26–50)<br>
<span style="color:#e67e22">■</span> High (51–75)<br>
<span style="color:#e74c3c">■</span> Critical (76–100)
</div>'''
risk_map.get_root().html.add_child(folium.Element(legend_html))

risk_map.save('../data/processed/chennai_risk_heatmap.html')
print('Heatmap saved: data/processed/chennai_risk_heatmap.html')
print('NOTE: This uses SIMULATED data. Run with live API for real scores.')
risk_map

<a id='section8'></a>
## 8. Time-of-Day Sensitivity Analysis

In [ ]:
# ── Time multiplier across all 24 hours ───────────────────────────────────────
hours = range(24)
multipliers = [get_time_multiplier(h).multiplier for h in hours]
band_names = [get_time_multiplier(h).band_name for h in hours]

band_colors = {
    'late_night': '#e74c3c',
    'early_morning': '#f39c12',
    'day': '#27ae60',
    'evening': '#e67e22',
}
colors_24 = [band_colors[b] for b in band_names]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: Multiplier by hour
bars = axes[0].bar(hours, multipliers, color=colors_24, alpha=0.85, edgecolor='white', linewidth=1)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Baseline (1.0)')
axes[0].set_xlabel('Hour of Day (24h)', fontsize=11)
axes[0].set_ylabel('Time Multiplier', fontsize=11)
axes[0].set_title('Explicit Time-of-Day Risk Multiplier\n(Same location, different risk at different hours)',
                  fontsize=12, fontweight='bold')
axes[0].set_xticks(hours)
axes[0].set_ylim(0.9, 1.5)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add band labels
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=b.replace('_',' ').title()) 
                   for b, c in band_colors.items()]
axes[0].legend(handles=legend_elements, loc='upper right', fontsize=9)

# Chart 2: How time multiplier affects a sample location's score
base_raw_score = 55  # Mid-risk location (before multiplier)
final_scores = [min(base_raw_score * m, 100) for m in multipliers]

axes[1].fill_between(hours, final_scores, alpha=0.3, color='#3498db')
axes[1].plot(hours, final_scores, color='#2980b9', linewidth=2.5, marker='o', markersize=5)
axes[1].axhline(y=75, color='#e74c3c', linestyle='--', alpha=0.5, label='High → Critical threshold')
axes[1].axhline(y=50, color='#f39c12', linestyle='--', alpha=0.5, label='Medium → High threshold')
axes[1].set_xlabel('Hour of Day (24h)', fontsize=11)
axes[1].set_ylabel('Final Risk Score', fontsize=11)
axes[1].set_title(f'Score Variation for a Mid-Risk Location (base={base_raw_score})\nacross 24 Hours',
                  fontsize=12, fontweight='bold')
axes[1].set_xticks(hours)
axes[1].set_ylim(45, 82)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/time_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTime multiplier range:', min(multipliers), '–', max(multipliers))
print('A score of 55 at midday becomes', round(55 * 1.40, 1), 'at 11pm (×1.40)')

<a id='section9'></a>
## 9. Example API Queries

In [ ]:
# ── Direct scoring engine query (no server needed) ────────────────────────────
# This demonstrates the scoring pipeline end-to-end.
# For production use, query via: GET /area-risk?lat=...&lng=...&hour=...

from app.services.macro_baseline_service import get_macro_baseline_score

def quick_score(lat, lng, hour, location_name,
                isolation=50.0, commercial=50.0,
                police=50.0, hospital=50.0, population=50.0):
    """Compute risk score with provided feature values (skips live API calls)."""
    from app.config import get_city_config
    city_config = get_city_config()
    h3_cell = latlng_to_h3(lat, lng, city_config.h3_resolution)
    time_band = get_time_multiplier(hour)
    crime_score, _, _ = get_macro_baseline_score()
    
    fs = FeatureSet(
        h3_cell=h3_cell, lat=lat, lng=lng, hour=hour,
        crime_baseline_score=crime_score,
        isolation_score=isolation,
        commercial_density_score=commercial,
        police_distance_score=police,
        hospital_distance_score=hospital,
        population_density_score=population,
        time_multiplier=time_band.multiplier,
        time_band_name=time_band.band_name,
        time_band_label=time_band.label,
        fallback_features=[]
    )
    scorer = WeightedRiskScorer()
    result = scorer.score(fs)
    
    print(f"{'='*55}")
    print(f"📍 {location_name}")
    print(f"   ({lat}, {lng}) | Hour: {hour:02d}:00 | H3: {h3_cell}")
    print(f"   Risk Score: {result.risk_score:.1f}/100 [{result.classification}]")
    print(f"   Time Band: {time_band.label} (×{time_band.multiplier})")
    print(f"   Contributing factors:")
    for k, v in result.contributing_factors.items():
        print(f"     {k}: {v:+.2f} pts")
    return result

print('\n=== EXAMPLE QUERIES (with pre-set feature values) ===\n')
print('These use the scoring formula directly. For live feature values,\n'
      'start the service and call GET /area-risk?lat=...&lng=...&hour=...\n')

# T. Nagar (busy commercial area, daytime)
r1 = quick_score(13.0418, 80.2341, 14, 'T. Nagar (Daytime)',
                 isolation=18, commercial=5, police=22, hospital=28, population=12)

# Vandalur isolated area, night
r2 = quick_score(12.8815, 80.0798, 22, 'Vandalur Isolated Stretch (Night)',
                 isolation=78, commercial=88, police=68, hospital=72, population=72)

<a id='section10'></a>
## 10. Validation & Conclusions

### Qualitative Validation Results

The model satisfies our expected qualitative ranking:

| Validation Check | Expected | Result |
|------------------|----------|--------|
| Isolated road > busy commercial road | ✓ | Vandalur > T. Nagar |
| Same location: night > day | ✓ | ×1.40 multiplier applied |
| Near police station < far away | ✓ | Sigmoid distance model |
| Dense commercial area < sparse area | ✓ | Inverted density score |

### Limitations (Honest Documentation)

1. **No ground truth**: We cannot compute Precision, Recall, or F1 — no labeled "this location is unsafe" dataset exists for Chennai. This is a structural limitation of data availability in India, not a modeling failure.

2. **NCRB data age**: The dataset is from 2014. Crime patterns may have changed, especially with Chennai's rapid development along OMR and Tambaram.

3. **Proxy signals**: Commercial density and road connectivity are proxies for safety, not direct measurements. A new residential area with few shops may score higher risk than its actual risk level.

4. **Weight subjectivity**: The scoring weights were set based on criminological literature (Jane Jacobs, routine activity theory). Without labeled training data, these cannot be empirically optimized — only qualitatively validated.

5. **Static population data**: WorldPop 2020 data does not capture intra-day population movement. A busy market area may have 0 population density in WorldPop but be very active during the day.

### Future Work

- Replace `WeightedRiskScorer` with `MLRiskScorer` trained on labeled safety survey data
- Integrate weather conditions as an additional multiplier
- Extend to other Indian cities via `city_config.yaml`
- Route safety scoring using the H3 neighbor graph (Phase 2)

---

### Data Sources Citation

- **NCRB**: National Crime Records Bureau (2014). *District-Wise Crime Statistics*. data.gov.in
- **OSM**: © OpenStreetMap contributors (ODbL). https://www.openstreetmap.org
- **WorldPop**: WorldPop, University of Southampton (2020). *Global High Resolution Population Denominators*. DOI: 10.5258/SOTON/WP00649
- **H3**: Uber Technologies (2018). *H3: Hexagonal Hierarchical Spatial Index*. https://h3geo.org
- **Jane Jacobs**: Jacobs, J. (1961). *The Death and Life of Great American Cities*. Random House.